In [7]:
%load_ext autoreload
%autoreload 2

```{toctree}
:maxdepth: 1
:caption: Contents:

# Walkthrough Example - Data Fetcher

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import xarray as xr

First, the different modules in the `oceanicospy.retrievals` package are imported.

In [18]:
from oceanicospy.retrievals import UHSLCDownloader,CMDSDownloader,ERA5Downloader

## University of Hawaii Sea Level Center (UHSLC) Data Downloader

This code is intended to download hourly sea-level data from the University of Hawaii Sea Level Center (UHSLC). The `UHSLCDownloader` class is defined to handle the downloading process. Only two parameters are required: the `station_id` for the station of interest and the `output_dir` where the downloaded data will be saved. The station id can be found on the UHSLC website (https://uhslc.soest.hawaii.edu/stations/):

In [21]:
UHSLCDownloader_obj = UHSLCDownloader(station_id='007', output_path='../data/temp/uhslc')

In [22]:
filepath = UHSLCDownloader_obj.download()

Downloaded h007.csv to ../data/temp/uhslc/h007.csv.


The method ``UHSLCDownloader.download`` returns ``filepath``, which is the path to the downloaded file. The file is saved in CSV format and can be read using pandas or any other data analysis tool. A short method is also included to quickly read the downloaded data into a pandas DataFrame ``UHSLCDownloader.clean_data``


In [23]:
df_clean = UHSLCDownloader_obj.clean_data(filepath)

In [24]:
df_clean

,depth[m]
1969-05-18 15:00:00,1.563
1969-05-18 16:00:00,1.399
1969-05-18 17:00:00,1.292
1969-05-18 18:00:00,1.277
1969-05-18 19:00:00,1.417
...,...
2026-01-31 19:00:00,1.316
2026-01-31 20:00:00,1.751
2026-01-31 21:00:00,2.073
2026-01-31 22:00:00,2.198


This data is not shifted to the local time zone, so the timestamps are in UTC.

# Copernicus Marine Data Store (CMDS) downloader

There is a class called `CMDSDownloader` that is designed to download data from the Copernicus Marine Data Store (CMDS). This class uses the `copernicusmarine` toolbox to submit spatial/temporal subset requests to the CMDS and download the resulting data. The class is initialized with the following parameters:
- `dataset_id`: The ID of the dataset to download (e.g., "dataset-id").
- `variables`: A list of variables to download (e.g., ["variable1", "variable2"]).
- spatial parameters for the region of interest (e.g., `lon_min`, `lon_max`, `lat_min`, `lat_max`).
- temporal parameters for the time range of interest (e.g., `start_date_local`, `end_date_local`).
- `difference_to_utc`: The time difference to UTC in hours (e.g., -5 for EST).
- `output_path`: The directory where the downloaded data will be saved.
- `file_format`: The format of the output file (e.g., "netcdf", "zarr").
- `filename`: The name of the output file (e.g., "output.nc").


## For winds
There is a classmethod that tailors the CMDSDownloader to download wind data from the CMDS. This method is called ``CMDSDownloader.for_winds`` and it takes the same parameters as the main class, but it automatically sets the `dataset_id` to the appropriate dataset for wind data and selects the relevant variables for wind analysis. This method simplifies the process of downloading wind data from the CMDS by pre-configuring the necessary parameters for this specific type of data.

For wind the following product and variables are defined as default:    
- Product: "cmems_obs-wind_glo_phy_nrt_l4_0.125deg_PT1H"
- Variables: ["eastward_wind", "northward_wind"]



In [16]:
winds_CMDS = CMDSDownloader.for_winds(
    lon_min=-82,
    lon_max=-81,
    lat_min=12,
    lat_max=13,
    start_datetime_local=datetime(2023, 12, 1, 0, 0),             
    end_datetime_local=datetime(2023, 12, 31, 23, 0),               
    difference_to_UTC=-5,                                       
    output_path="../data/temp/CMDS",
    output_filename="winds_CMDS.nc",                             
    file_format="netcdf",
)

In [ ]:
winds_CMDS_filepath = winds_CMDS.download()

INFO - 2026-03-30T02:29:31Z - Selected dataset version: "202207"
INFO:copernicusmarine:Selected dataset version: "202207"
INFO - 2026-03-30T02:29:31Z - Selected dataset part: "default"
INFO:copernicusmarine:Selected dataset part: "default"


ImportError: No module named 'h5py', backend not available. Please install 'h5py' into your Python environment.

In [ ]:
winds_CMDS_ds = xr.open_dataset(winds_CMDS_filepath) 
winds_CMDS_ds

## For waves

Similar to winds, there is a default configuration for downloading wave data from the CMDS. The classmethod ``CMDSDownloader.for_waves`` is designed to facilitate the download of wave data by pre-setting the appropriate `dataset_id` and selecting the relevant variables for wave analysis. This method allows users to easily access wave data from the CMDS without needing to manually configure the parameters.

For waves the following product and variables are defined as default:
- Product: "cmems_obs-wave_glo_wav_anfc_0.083deg_PT3H"
- Variables: ["VHM0", "VMDR", "VTPK"]

In [14]:
waves_CMDS = CMDSDownloader.for_waves(
    lon_min=-82,
    lon_max=-81,
    lat_min=12,
    lat_max=13,
    start_datetime_local=datetime(2023, 8, 1, 0, 0),             
    end_datetime_local=datetime(2023, 8, 31, 23, 0),               
    difference_to_UTC=-5,                                       
    output_path="../data/temp/CMDS",
    output_filename="waves_CMDS.nc",                             
    file_format="netcdf",
)

In [ ]:
waves_CMDS_filepath = waves_CMDS.download()

INFO - 2026-03-30T02:28:12Z - Selected dataset version: "202411"
INFO - 2026-03-30T02:28:12Z - Selected dataset part: "default"


ImportError: No module named 'h5py', backend not available. Please install 'h5py' into your Python environment.

In [ ]:
waves_CMDS_ds = xr.open_dataset(waves_CMDS_filepath) 
waves_CMDS_ds

# ERA5 Downloader

ERA5 is the fifth generation ECMWF reanalysis for the global climate and weather. This product is provided by the European Centre for Medium-Range Weather Forecasts (ECMWF) and is available through the Copernicus Climate Data Store (CDS). The `ERA5Downloader` class is designed to facilitate the download of ERA5 data from the CDS. This class uses the `cdsapi` library to submit requests to the CDS and download the resulting data.

In [19]:
variables = ["10m_u_component_of_wind","10m_v_component_of_wind",]

# Spatial domain (WGS84); San Andrés example
lon_min, lon_max = -81.90, -81.50   # West/East
lat_min, lat_max =  12.35,  12.75   # South/North

start_local = datetime(2023, 8, 1, 0, 0)
end_local   = datetime(2023, 8, 31, 23, 0)
utc_offset_hours = -5  # Local = UTC-5

# Output file
output_path = "../data/temp/ERA5/"

winds_ERA5 = ERA5Downloader(
    variables=variables,
    lon_min=lon_min,
    lon_max=lon_max,
    lat_min=lat_min,
    lat_max=lat_max,
    start_datetime_local=start_local,
    end_datetime_local=end_local,
    difference_to_UTC=utc_offset_hours,
    output_path=output_path,
    output_filename="winds_ERA5.nc",
)

In [20]:
winds_ERA5_filepath = winds_ERA5.download()

/Users/franklinayala/venvs/oceanicospy-dev-ffayalac/lib/python3.10/site-packages/ecmwf/datastores/client.py:87: UserWarning: Invalid URL 'None/catalogue/v1/messages': No scheme supplied. Perhaps you meant https://None/catalogue/v1/messages?
  warnings.warn(str(exc), UserWarning)


MissingSchema: Invalid URL 'None/retrieve/v1/processes/reanalysis-era5-single-levels': No scheme supplied. Perhaps you meant https://None/retrieve/v1/processes/reanalysis-era5-single-levels?

In [ ]:
winds_ERA5.format_to_localtime()